**1. Import Required Libraries**

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**2. Load the 5G NIDD Dataset**

In [ ]:
# Load
df = pd.read_csv('/content/drive/MyDrive/5G_NIDD_multiclass_clean.csv', low_memory=False)

print("Original shape:", df.shape)

# Target
y = df['Label']
X = df.drop(columns=['Label', 'Attack Type', 'Attack Tool'], errors='ignore')

# Remove obvious non-learning columns
drop_cols = [
    'SrcMac','DstMac','SrcAddr','DstAddr','StartTime','LastTime',        # drop these columns as they dont have any importance in model(these are IP addresses)
    'SrcOui','DstOui'
]

X = X.drop(columns=[c for c in drop_cols if c in X.columns], errors='ignore')

# Keep only numeric features
X = X.select_dtypes(include=[np.number])   # keeps only numeric features

print("After numeric selection:", X.shape)

Original shape: (1215890, 112)
After numeric selection: (1215890, 86)


**3. Data Cleaning and Handling Missing Values**

In [ ]:
X.replace([np.inf, -np.inf], np.nan, inplace=True)     # remove infinite values
X.fillna(0, inplace=True)                              # handle missing values

**4. Feature Selection using SelectKBest**

In [ ]:
selector = SelectKBest(score_func=f_classif, k=36)
X_selected = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]
print("Selected Features:", selected_features.tolist())

/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [ 1  7 13 14 19 20 21 22 23 24 25 26 27 34 35 36 37 48 49 50 51 57 58 59
 60 61 62 63 64 65 66 67 68 69 70 71 72 77 78 79 80] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


Selected Features: ['Rank', 'Seq', 'Dur', 'RunTime', 'Mean', 'Sum', 'Min', 'Max', 'sTos', 'dTos', 'sTtl', 'dTtl', 'sHops', 'dHops', 'TotPkts', 'SrcPkts', 'DstPkts', 'TotBytes', 'SrcBytes', 'DstBytes', 'Offset', 'sMeanPktSz', 'dMeanPktSz', 'Loss', 'SrcLoss', 'DstLoss', 'pLoss', 'SrcWin', 'DstWin', 'sVid', 'dVid', 'SrcTCPBase', 'DstTCPBase', 'TcpRtt', 'SynAck', 'AckDat']


**5. Label Encoding(Encode Target Labels)**

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

num_classes = len(np.unique(y_encoded))
print("Classes:", num_classes)

Classes: 20


**6. Split Dataset into Training and Testing Sets**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y_encoded,
    test_size=0.2,                    # train data = 80%, test data = 20%
    stratify=y_encoded,
    random_state=42
)

**7. Feature Scaling using QuantileTransformer**

In [ ]:
from sklearn.preprocessing import QuantileTransformer

scaler = QuantileTransformer(
    n_quantiles=min(1000, len(X_train)),   # avoids warning if dataset < 1000
    output_distribution='normal',          # 'normal' usually works better for neural networks
    random_state=42
)

X_train = scaler.fit_transform(X_train)   # FIT ONLY TRAIN
X_test  = scaler.transform(X_test)        # TRANSFORM TEST

**8. Reshape Data for BiGRU Input**

In [ ]:
X_train = X_train.reshape(-1, 36, 1)        # Prepare data in 3D format required by GRU networks
X_test  = X_test.reshape(-1, 36, 1)

**9. Import Deep Learning Layers and Mobile-Net-V1 & BIGRU(Combined) MODEL**

In [ ]:
def MobileNetV1_BiGRU(drop_rate, gru_units, dense_units):

    inp = Input(shape=(36,1))

    # ----- CNN BRANCH -----                    # MOBILE-NET V1 MODEL
    x = Reshape((36,1,1))(inp)

    x = Conv2D(32,(3,3),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = DepthwiseConv2D((3,3),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Conv2D(64,(1,1),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = DepthwiseConv2D((3,3),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Conv2D(128,(1,1),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    cnn_out = GlobalAveragePooling2D()(x)

    # ----- GRU BRANCH -----                                 # BIGRU MODEL
    r = Bidirectional(GRU(gru_units, return_sequences=True))(inp)
    r = Bidirectional(GRU(gru_units))(r)

    # ----- FUSION -----                                    # PROJECTION(Combines both models)
    merged = Concatenate()([cnn_out, r])

    merged = Dense(dense_units, activation="relu")(merged)
    merged = Dense(dense_units//2, activation="relu")(merged)
    merged = Dropout(drop_rate)(merged)

    out = Dense(num_classes, activation="softmax")(merged)

    model = Model(inp, out)
    return model


**10. Install keras tuner**

In [ ]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 6.8 MB/s eta 0:00:00


In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 13.9 MB/s eta 0:00:00


**11. Setup TensorFlow & Focal Loss**

In [ ]:
import tensorflow as tf

def focal_loss(gamma, alpha):

    def loss(y_true, y_pred):

        y_true = tf.cast(y_true, tf.int32)
        y_true = tf.one_hot(y_true, depth=num_classes)

        ce = tf.keras.losses.categorical_crossentropy(y_true, y_pred)

        pt = tf.exp(-ce)
        focal = alpha * tf.pow(1 - pt, gamma) * ce

        return tf.reduce_mean(focal)

    return loss


**12. Handle Class Imbalance using Class Weights**

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, class_weights))
print(class_weights)


{np.int64(0): np.float64(0.64811172410117), np.int64(1): np.float64(0.6491671115856914), np.int64(2): np.float64(3.325965944060726), np.int64(3): np.float64(4.20650406504065), np.int64(4): np.float64(37.819284603421465), np.int64(5): np.float64(44.74296228150874), np.int64(6): np.float64(1.3619983757596124), np.int64(7): np.float64(4.309374446216552), np.int64(8): np.float64(5.309563318777292), np.int64(9): np.float64(5.274438781043271), np.int64(10): np.float64(1.9601644365629534), np.int64(11): np.float64(4.803516049382716), np.int64(12): np.float64(5.220652640618291), np.int64(13): np.float64(5.217292426517915), np.int64(14): np.float64(1.5948189926547744), np.int64(15): np.float64(1.9197000197355436), np.int64(16): np.float64(0.12998123867505493), np.int64(17): np.float64(0.21242149215139894), np.int64(18): np.float64(6.053721682847897), np.int64(19): np.float64(5.8995147986414365)}


**Dividing data further**

100% DATA = 80% TRAIN | 20% TEST   (test=used for final evaluation)

TRAIN = 80% CLIENTS | 20% SERVER   (server=used for each round evaluation)

CLIENT = 70% FL DATA | 30% PRETRAIN  (fl=hypertuning and divided b/w clients)

In [ ]:
from sklearn.model_selection import train_test_split

# Train-Test split
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

# Server (20%) and Client (80%)
X_server, X_clients, y_server, y_clients = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.8,
    stratify=y_train_full,
    random_state=42
)

# # Split client data → FL + (optional small pretrain if needed later)
# X_fl, _, y_fl, _ = train_test_split(
#     X_clients,
#     y_clients,
#     test_size=0.3,
#     stratify=y_clients,
#     random_state=42
# )

**13. Optuna Training of hyperparameters**

In [ ]:
import optuna
import tensorflow as tf
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

X_tune = X_clients
y_tune = y_clients

BEST_MODEL_PATH = "best_optuna_model.weights.h5"
best_score = 0

def objective(trial):
    global best_score
    print(f"\nStarting Trial {trial.number}")

    tf.keras.backend.clear_session()

    dropout = trial.suggest_float("dropout", 0.2, 0.4, step=0.1)
    gru_units = trial.suggest_int("gru_units", 64, 192, step=64)
    dense_units = trial.suggest_int("dense_units", 256, 384, step=64)
    gamma = trial.suggest_float("gamma", 1.0, 3.0, step=1)
    alpha = trial.suggest_float("alpha", 0.2, 0.4, step=0.1)
    lr = trial.suggest_float("lr", 1e-4, 1e-3, log=True)
    epochs = trial.suggest_int("epochs", 10, 20, step=10)

    model = MobileNetV1_BiGRU(
        drop_rate=dropout,
        gru_units=gru_units,
        dense_units=dense_units
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=focal_loss(gamma=gamma, alpha=alpha),
        metrics=["accuracy"]
    )

    history = model.fit(
        X_tune,
        y_tune,
        validation_split=0.2,
        epochs=epochs,
        batch_size=256,
        callbacks=[
            EarlyStopping(patience=3, restore_best_weights=True),
            ReduceLROnPlateau(patience=2)
        ],
        verbose=1
    )

    val_acc = max(history.history["val_accuracy"])

    if val_acc > best_score:
        best_score = val_acc
        model.save_weights(BEST_MODEL_PATH)

    return val_acc


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=6)

best_params = study.best_params

[I 2026-04-18 13:17:57,606] A new study created in memory with name: no-name-a75e8414-56cc-4cc7-a40b-2e7962cc1ed8



Starting Trial 0
Epoch 1/20
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 93s 42ms/step - accuracy: 0.7528 - loss: 0.0958 - val_accuracy: 0.8669 - val_loss: 0.0441 - learning_rate: 3.6304e-04
Epoch 2/20
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 83s 43ms/step - accuracy: 0.8673 - loss: 0.0399 - val_accuracy: 0.8679 - val_loss: 0.0343 - learning_rate: 3.6304e-04
Epoch 3/20
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 82s 42ms/step - accuracy: 0.8843 - loss: 0.0324 - val_accuracy: 0.8845 - val_loss: 0.0288 - learning_rate: 3.6304e-04
Epoch 4/20
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 82s 42ms/step - accuracy: 0.8920 - loss: 0.0290 - val_accuracy: 0.9023 - val_loss: 0.0258 - learning_rate: 3.6304e-04
Epoch 5/20
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 82s 42ms/step - accuracy: 0.8975 - loss: 0.0268 - val_accuracy: 0.8984 - val_loss: 0.0263 - learning_rate: 3.6304e-04
Epoch 6/20
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 83s 42ms/step - accuracy: 0.9020 - loss: 0.0250 - val_accuracy: 0.9169 - val_loss: 0.0222 - learning_rate: 3.6304e-04
Epoch 7/20
1946/1946 ━

[I 2026-04-18 13:45:36,540] Trial 0 finished with value: 0.9530311226844788 and parameters: {'dropout': 0.2, 'gru_units': 192, 'dense_units': 320, 'gamma': 2.0, 'alpha': 0.30000000000000004, 'lr': 0.00036303852518457084, 'epochs': 20}. Best is trial 0 with value: 0.9530311226844788.



Starting Trial 1
Epoch 1/20
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 65s 30ms/step - accuracy: 0.6853 - loss: 0.2267 - val_accuracy: 0.8000 - val_loss: 0.1288 - learning_rate: 1.8611e-04
Epoch 2/20
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 57s 30ms/step - accuracy: 0.8249 - loss: 0.1105 - val_accuracy: 0.8573 - val_loss: 0.0868 - learning_rate: 1.8611e-04
Epoch 3/20
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 58s 30ms/step - accuracy: 0.8618 - loss: 0.0835 - val_accuracy: 0.8840 - val_loss: 0.0697 - learning_rate: 1.8611e-04
Epoch 4/20
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 58s 30ms/step - accuracy: 0.8749 - loss: 0.0723 - val_accuracy: 0.8941 - val_loss: 0.0640 - learning_rate: 1.8611e-04
Epoch 5/20
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 58s 30ms/step - accuracy: 0.8828 - loss: 0.0664 - val_accuracy: 0.8867 - val_loss: 0.0624 - learning_rate: 1.8611e-04
Epoch 6/20
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 58s 30ms/step - accuracy: 0.8889 - loss: 0.0619 - val_accuracy: 0.8932 - val_loss: 0.0591 - learning_rate: 1.8611e-04
Epoch 7/20
1946/1946 ━

[I 2026-04-18 14:05:26,998] Trial 1 finished with value: 0.9407749176025391 and parameters: {'dropout': 0.30000000000000004, 'gru_units': 128, 'dense_units': 256, 'gamma': 1.0, 'alpha': 0.4, 'lr': 0.00018611105596172522, 'epochs': 20}. Best is trial 0 with value: 0.9530311226844788.


...

[I 2026-04-18 14:30:24,068] Trial 3 finished with value: 0.9110579490661621 and parameters: {'dropout': 0.4, 'gru_units': 64, 'dense_units': 320, 'gamma': 3.0, 'alpha': 0.30000000000000004, 'lr': 0.00045141016419007605, 'epochs': 10}. Best is trial 2 with value: 0.9615285992622375.



Starting Trial 4
Epoch 1/20
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 89s 42ms/step - accuracy: 0.7108 - loss: 0.1478 - val_accuracy: 0.8212 - val_loss: 0.0790 - learning_rate: 2.7848e-04
Epoch 2/20
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 83s 42ms/step - accuracy: 0.8388 - loss: 0.0691 - val_accuracy: 0.8657 - val_loss: 0.0542 - learning_rate: 2.7848e-04
Epoch 3/20
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 82s 42ms/step - accuracy: 0.8720 - loss: 0.0513 - val_accuracy: 0.8921 - val_loss: 0.0425 - learning_rate: 2.7848e-04
Epoch 4/20
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 81s 42ms/step - accuracy: 0.8830 - loss: 0.0445 - val_accuracy: 0.8959 - val_loss: 0.0382 - learning_rate: 2.7848e-04
Epoch 5/20
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 82s 42ms/step - accuracy: 0.8899 - loss: 0.0407 - val_accuracy: 0.9077 - val_loss: 0.0351 - learning_rate: 2.7848e-04
Epoch 6/20
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 82s 42ms/step - accuracy: 0.8937 - loss: 0.0381 - val_accuracy: 0.9035 - val_loss: 0.0342 - learning_rate: 2.7848e-04
Epoch 7/20
1946/1946 ━

[I 2026-04-18 14:57:55,027] Trial 4 finished with value: 0.9318597912788391 and parameters: {'dropout': 0.30000000000000004, 'gru_units': 192, 'dense_units': 320, 'gamma': 2.0, 'alpha': 0.4, 'lr': 0.0002784752529980659, 'epochs': 20}. Best is trial 2 with value: 0.9615285992622375.



Starting Trial 5
Epoch 1/20
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 65s 30ms/step - accuracy: 0.6723 - loss: 0.0750 - val_accuracy: 0.7733 - val_loss: 0.0400 - learning_rate: 1.6707e-04
Epoch 2/20
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 58s 30ms/step - accuracy: 0.8102 - loss: 0.0324 - val_accuracy: 0.8526 - val_loss: 0.0245 - learning_rate: 1.6707e-04
Epoch 3/20
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 58s 30ms/step - accuracy: 0.8557 - loss: 0.0235 - val_accuracy: 0.8586 - val_loss: 0.0229 - learning_rate: 1.6707e-04
Epoch 4/20
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 58s 30ms/step - accuracy: 0.8701 - loss: 0.0198 - val_accuracy: 0.8821 - val_loss: 0.0173 - learning_rate: 1.6707e-04
Epoch 5/20
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 82s 30ms/step - accuracy: 0.8791 - loss: 0.0177 - val_accuracy: 0.8877 - val_loss: 0.0159 - learning_rate: 1.6707e-04
Epoch 6/20
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 58s 30ms/step - accuracy: 0.8864 - loss: 0.0161 - val_accuracy: 0.8886 - val_loss: 0.0148 - learning_rate: 1.6707e-04
Epoch 7/20
1946/1946 ━

[I 2026-04-18 15:18:06,457] Trial 5 finished with value: 0.9342933893203735 and parameters: {'dropout': 0.2, 'gru_units': 128, 'dense_units': 256, 'gamma': 3.0, 'alpha': 0.2, 'lr': 0.00016706683104907827, 'epochs': 20}. Best is trial 2 with value: 0.9615285992622375.


**14. Compiled model**

In [ ]:
def get_compiled_model():
    model = MobileNetV1_BiGRU(
        drop_rate=0.2,
        gru_units=192,
        dense_units=320
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0009661918592170282),
        loss=focal_loss(gamma=1.0, alpha=0.4),
        metrics=["accuracy"]
    )

    return model

In [ ]:
BEST_MODEL_PATH = "best_optuna_model.weights.h5"
global_model = get_compiled_model()
global_model.load_weights(BEST_MODEL_PATH)

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 78 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


**15. Create Federated Clients**

In [ ]:
from sklearn.model_selection import StratifiedKFold

NUM_CLIENTS = 7

def create_clients(X, y, num_clients):
    skf = StratifiedKFold(n_splits=num_clients, shuffle=True, random_state=42)
    client_data = {}

    for i, (_, idx) in enumerate(skf.split(X, y)):
        client_data[i] = (X[idx], y[idx])

    return client_data


client_data = create_clients(X_clients, y_clients, NUM_CLIENTS)

**16. Client Local Training Function**

In [ ]:
def client_update(global_weights, dataset, local_epochs=2):

    local_model = get_compiled_model()
    local_model.set_weights(global_weights)

    X, y = dataset

    local_model.fit(
        X,
        y,
        epochs=local_epochs,
        batch_size=256,
        verbose=0
    )

    return local_model.get_weights(), len(X)

**17. Federated Averaging (FedAvg)**

In [ ]:
def aggregate_weights(client_weights, client_sizes):

    total_samples = sum(client_sizes)
    avg_weights = []

    for weights_list_tuple in zip(*client_weights):
        weighted_sum = np.zeros_like(weights_list_tuple[0])

        for w, size in zip(weights_list_tuple, client_sizes):
            weighted_sum += w * (size / total_samples)

        avg_weights.append(weighted_sum)

    return avg_weights

**18. Federated Training Loop (Server)**

In [ ]:
import random

ROUNDS = 13
CLIENTS_PER_ROUND = 3

for round_num in range(ROUNDS):

    print(f"\nFL Round {round_num}")

    global_weights = global_model.get_weights()

    client_weights = []
    client_sizes = []

    selected_clients = random.sample(
        list(client_data.keys()),
        CLIENTS_PER_ROUND
    )

    for client in selected_clients:
        weights, size = client_update(
            global_weights,
            client_data[client]
        )
        client_weights.append(weights)
        client_sizes.append(size)

    new_global_weights = aggregate_weights(client_weights, client_sizes)

    global_model.set_weights(new_global_weights)

    # Server evaluation (Approach 2)
    loss, acc = global_model.evaluate(X_server, y_server, verbose=1)
    print(f"Server Accuracy: {acc:.4f}")

**19. Final Accuracy**

In [ ]:
loss, acc = global_model.evaluate(X_test, y_test, verbose=1)
print("\nFinal Test Accuracy:", acc)

6080/6080 ━━━━━━━━━━━━━━━━━━━━ 58s 10ms/step - accuracy: 0.9543 - loss: 0.0263

Final Test Accuracy: 0.9542620182037354


In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)
import numpy as np

# 🔹 Get predictions
y_pred_probs = global_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# 🔹 If y_test is one-hot encoded
if len(y_test.shape) > 1:
    y_true = np.argmax(y_test, axis=1)
else:
    y_true = y_test

# 🔹 Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:\n", cm)

# 🔹 Classification Report (precision, recall, f1 per class)
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))

# 🔹 Overall Metrics
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print(f"\nPrecision: {precision:.4f}")
print(f"Recall (TPR): {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

6080/6080 ━━━━━━━━━━━━━━━━━━━━ 38s 6ms/step

Confusion Matrix:
 [[14833    22     5     6     8     3    37     0     0     0    10    23
      0     6    30    12     7     7     0     0]
 [    4 14736    11    18     6    22    18     5     3     0    89    19
      1     5     5     2     5    34     0     1]
 [   49     7  2699    27     1     2    39     3     0     1     3     5
      0     7    12     0    29     9    25     7]
 [    0    16    50  2134     9    22     2     5     0     1    11     5
      0     0     3     1    20    26     0     7]
 [    1     0    11    28   138    60     0     5     0     1     1     1
      0     0     0     0     5     5     0     1]
 [    0     2     0     9    15   171     0     7     0     0     1     1
      0     1     0     8     0     1     0     1]
 [    1     3    31     6    11     8  6936     7     0    12    36    38
      0     9     7     5     8    23     1     0]
 [    0     0     0     1     0     0    17  2191     0     1

In [ ]:
# For binary classification only
if len(np.unique(y_true)) == 2:

    tn, fp, fn, tp = cm.ravel()

    tpr = tp / (tp + fn)   # Recall / Sensitivity
    fpr = fp / (fp + tn)

    print(f"\nTPR (Recall): {tpr:.4f}")
    print(f"FPR: {fpr:.4f}")

In [ ]:
num_classes = cm.shape[0]

for i in range(num_classes):
    tp = cm[i, i]
    fn = np.sum(cm[i, :]) - tp
    fp = np.sum(cm[:, i]) - tp
    tn = np.sum(cm) - (tp + fn + fp)

    tpr = tp / (tp + fn) if (tp + fn) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0

    print(f"\nClass {i}: TPR={tpr:.4f}, FPR={fpr:.4f}")


Class 0: TPR=0.9883, FPR=0.0007

Class 1: TPR=0.9834, FPR=0.0013

Class 2: TPR=0.9227, FPR=0.0014

Class 3: TPR=0.9230, FPR=0.0012

Class 4: TPR=0.5370, FPR=0.0004

Class 5: TPR=0.7880, FPR=0.0010

Class 6: TPR=0.9712, FPR=0.0013

Class 7: TPR=0.9708, FPR=0.0003

Class 8: TPR=0.9301, FPR=0.0008

Class 9: TPR=0.8769, FPR=0.0008

Class 10: TPR=0.9619, FPR=0.0030

Class 11: TPR=0.7699, FPR=0.0014

Class 12: TPR=0.7370, FPR=0.0017

Class 13: TPR=0.7677, FPR=0.0024

Class 14: TPR=0.9497, FPR=0.0005

Class 15: TPR=0.9882, FPR=0.0004

Class 16: TPR=0.9459, FPR=0.0080

Class 17: TPR=0.9767, FPR=0.0272

Class 18: TPR=0.9564, FPR=0.0013

Class 19: TPR=0.9115, FPR=0.0005


In [ ]:
import random

ROUNDS = 13
CLIENTS_PER_ROUND = 5

for round_num in range(ROUNDS):

    print(f"\nFL Round {round_num}")

    global_weights = global_model.get_weights()

    client_weights = []
    client_sizes = []

    selected_clients = random.sample(
        list(client_data.keys()),
        CLIENTS_PER_ROUND
    )

    for client in selected_clients:
        weights, size = client_update(
            global_weights,
            client_data[client]
        )
        client_weights.append(weights)
        client_sizes.append(size)

    new_global_weights = aggregate_weights(client_weights, client_sizes)

    global_model.set_weights(new_global_weights)

    # Server evaluation (Approach 2)
    loss, acc = global_model.evaluate(X_server, y_server, verbose=1)
    print(f"Server Accuracy: {acc:.4f}")


FL Round 0
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 34s 7ms/step - accuracy: 0.9744 - loss: 0.0203
Server Accuracy: 0.9744

FL Round 1
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 36s 7ms/step - accuracy: 0.9730 - loss: 0.0207
Server Accuracy: 0.9730

FL Round 2
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 34s 7ms/step - accuracy: 0.9668 - loss: 0.0214
Server Accuracy: 0.9668

FL Round 3
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 35s 7ms/step - accuracy: 0.9734 - loss: 0.0204
Server Accuracy: 0.9734

FL Round 4
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 36s 7ms/step - accuracy: 0.9700 - loss: 0.0212
Server Accuracy: 0.9700

FL Round 5
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 34s 7ms/step - accuracy: 0.9752 - loss: 0.0194
Server Accuracy: 0.9752

FL Round 6
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 35s 7ms/step - accuracy: 0.9768 - loss: 0.0197
Server Accuracy: 0.9768

FL Round 7
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 34s 7ms/step - accuracy: 0.9710 - loss: 0.0209
Server Accuracy: 0.9710

FL Round 8
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 35s 7ms/step - accuracy: 0.9698 - loss: 0.022

In [ ]:
loss, acc = global_model.evaluate(X_test, y_test, verbose=1)
print("\nFinal Test Accuracy:", acc)

6080/6080 ━━━━━━━━━━━━━━━━━━━━ 43s 7ms/step - accuracy: 0.9770 - loss: 0.0189

Final Test Accuracy: 0.9770385026931763


In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)
import numpy as np

# 🔹 Get predictions
y_pred_probs = global_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# 🔹 If y_test is one-hot encoded
if len(y_test.shape) > 1:
    y_true = np.argmax(y_test, axis=1)
else:
    y_true = y_test

# 🔹 Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:\n", cm)

# 🔹 Classification Report (precision, recall, f1 per class)
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))

# 🔹 Overall Metrics
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print(f"\nPrecision: {precision:.4f}")
print(f"Recall (TPR): {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

6080/6080 ━━━━━━━━━━━━━━━━━━━━ 50s 8ms/step

Confusion Matrix:
 [[14790    21     5     7     8     1    41     0     0     0     8    23
      0     3    54     6    30    12     0     0]
 [    4 14727     8    23     9    13    10     6     1     0    37    79
      0     2     2     1    13    42     0     7]
 [    5     6  2678    49     2     0    41     4     0     1     3     8
      0     1    82     0    29     9     6     1]
 [    0    19    15  2168    10     5     4     4     0     0     5    12
      1     3     7     1    20    18     0    20]
 [    0     0    11    15   155    27     2     7     0     0     0     0
      0     0     0     0    17     8     0    15]
 [    0     0     0    35    29   134     0     0     0     0     2     0
      0     0     0     5     2     6     0     4]
 [    0     5    36    11    17     1  6977    10     0     1    20    17
      0     6     4     6    11    20     0     0]
 [    1     1     0     0     0     1    13  2197     0     1

In [ ]:
# For binary classification only
if len(np.unique(y_true)) == 2:

    tn, fp, fn, tp = cm.ravel()

    tpr = tp / (tp + fn)   # Recall / Sensitivity
    fpr = fp / (fp + tn)

    print(f"\nTPR (Recall): {tpr:.4f}")
    print(f"FPR: {fpr:.4f}")

In [ ]:
num_classes = cm.shape[0]

for i in range(num_classes):
    tp = cm[i, i]
    fn = np.sum(cm[i, :]) - tp
    fp = np.sum(cm[:, i]) - tp
    tn = np.sum(cm) - (tp + fn + fp)

    tpr = tp / (tp + fn) if (tp + fn) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0

    print(f"\nClass {i}: TPR={tpr:.4f}, FPR={fpr:.4f}")


Class 0: TPR=0.9854, FPR=0.0002

Class 1: TPR=0.9828, FPR=0.0009

Class 2: TPR=0.9156, FPR=0.0011

Class 3: TPR=0.9377, FPR=0.0015

Class 4: TPR=0.6031, FPR=0.0006

Class 5: TPR=0.6175, FPR=0.0004

Class 6: TPR=0.9769, FPR=0.0015

Class 7: TPR=0.9734, FPR=0.0003

Class 8: TPR=0.9290, FPR=0.0006

Class 9: TPR=0.8839, FPR=0.0003

Class 10: TPR=0.8995, FPR=0.0012

Class 11: TPR=0.8844, FPR=0.0037

Class 12: TPR=0.7729, FPR=0.0012

Class 13: TPR=0.8530, FPR=0.0018

Class 14: TPR=0.9567, FPR=0.0008

Class 15: TPR=0.9876, FPR=0.0003

Class 16: TPR=0.9930, FPR=0.0032

Class 17: TPR=0.9915, FPR=0.0041

Class 18: TPR=0.9371, FPR=0.0010

Class 19: TPR=0.9557, FPR=0.0009


In [ ]:
import random

ROUNDS = 13
CLIENTS_PER_ROUND = 7

for round_num in range(ROUNDS):

    print(f"\nFL Round {round_num}")

    global_weights = global_model.get_weights()

    client_weights = []
    client_sizes = []

    selected_clients = random.sample(
        list(client_data.keys()),
        CLIENTS_PER_ROUND
    )

    for client in selected_clients:
        weights, size = client_update(
            global_weights,
            client_data[client]
        )
        client_weights.append(weights)
        client_sizes.append(size)

    new_global_weights = aggregate_weights(client_weights, client_sizes)

    global_model.set_weights(new_global_weights)

    # Server evaluation (Approach 2)
    loss, acc = global_model.evaluate(X_server, y_server, verbose=1)
    print(f"Server Accuracy: {acc:.4f}")


FL Round 0
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 37s 7ms/step - accuracy: 0.9319 - loss: 0.0335
Server Accuracy: 0.9319

FL Round 1
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 36s 7ms/step - accuracy: 0.9283 - loss: 0.0351
Server Accuracy: 0.9283

FL Round 2
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 37s 8ms/step - accuracy: 0.9273 - loss: 0.0360
Server Accuracy: 0.9273

FL Round 3
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 36s 7ms/step - accuracy: 0.9397 - loss: 0.0331
Server Accuracy: 0.9397

FL Round 4
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 35s 7ms/step - accuracy: 0.9313 - loss: 0.0344
Server Accuracy: 0.9313

FL Round 5
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 36s 7ms/step - accuracy: 0.9328 - loss: 0.0339
Server Accuracy: 0.9328

FL Round 6
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 36s 7ms/step - accuracy: 0.9406 - loss: 0.0321
Server Accuracy: 0.9406

FL Round 7
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 36s 7ms/step - accuracy: 0.9389 - loss: 0.0334
Server Accuracy: 0.9389

FL Round 8
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 35s 7ms/step - accuracy: 0.9388 - loss: 0.031

In [ ]:
loss, acc = global_model.evaluate(X_test, y_test, verbose=1)
print("\nFinal Test Accuracy:", acc)

6080/6080 ━━━━━━━━━━━━━━━━━━━━ 46s 8ms/step - accuracy: 0.9477 - loss: 0.0295

Final Test Accuracy: 0.9477390646934509


In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)
import numpy as np

# 🔹 Get predictions
y_pred_probs = global_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# 🔹 If y_test is one-hot encoded
if len(y_test.shape) > 1:
    y_true = np.argmax(y_test, axis=1)
else:
    y_true = y_test

# 🔹 Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:\n", cm)

# 🔹 Classification Report (precision, recall, f1 per class)
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))

# 🔹 Overall Metrics
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print(f"\nPrecision: {precision:.4f}")
print(f"Recall (TPR): {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

6080/6080 ━━━━━━━━━━━━━━━━━━━━ 32s 5ms/step

Confusion Matrix:
 [[14756    45    10     7     6     5    42     1     0     0    10    23
      0     4    53    12    29     6     0     0]
 [    3 14678     5    35     3    18    27     4     0     0    90    52
      1     3     2     2    22    32     0     7]
 [   13     8  2704    49     5     0    33     4     0     0     2     6
      0     2    66     0    25     6     0     2]
 [    0    16    31  2157     4    15     1     6     0     0     2     8
      0     1     1     0    28    21     0    21]
 [    0     0    10    29   111    59     1     7     0     0     1     1
      0     1     0     0    14     3     0    20]
 [    0     2     0    19     9   173     0     0     0     0     1     3
      2     0     0     2     1     2     0     3]
 [    0     8    49     6     6    10  6947     5     0     3    22    29
      1     3     7     7    18    20     1     0]
 [    0     2     1     4     0     0    16  2197     0     1

In [ ]:
# For binary classification only
if len(np.unique(y_true)) == 2:

    tn, fp, fn, tp = cm.ravel()

    tpr = tp / (tp + fn)   # Recall / Sensitivity
    fpr = fp / (fp + tn)

    print(f"\nTPR (Recall): {tpr:.4f}")
    print(f"FPR: {fpr:.4f}")

In [ ]:
num_classes = cm.shape[0]

for i in range(num_classes):
    tp = cm[i, i]
    fn = np.sum(cm[i, :]) - tp
    fp = np.sum(cm[:, i]) - tp
    tn = np.sum(cm) - (tp + fn + fp)

    tpr = tp / (tp + fn) if (tp + fn) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0

    print(f"\nClass {i}: TPR={tpr:.4f}, FPR={fpr:.4f}")


Class 0: TPR=0.9831, FPR=0.0002

Class 1: TPR=0.9796, FPR=0.0013

Class 2: TPR=0.9244, FPR=0.0015

Class 3: TPR=0.9330, FPR=0.0018

Class 4: TPR=0.4319, FPR=0.0003

Class 5: TPR=0.7972, FPR=0.0008

Class 6: TPR=0.9727, FPR=0.0017

Class 7: TPR=0.9734, FPR=0.0004

Class 8: TPR=0.9176, FPR=0.0004

Class 9: TPR=0.8905, FPR=0.0003

Class 10: TPR=0.9486, FPR=0.0027

Class 11: TPR=0.7511, FPR=0.0021

Class 12: TPR=0.7225, FPR=0.0013

Class 13: TPR=0.8369, FPR=0.0026

Class 14: TPR=0.9567, FPR=0.0008

Class 15: TPR=0.9862, FPR=0.0003

Class 16: TPR=0.9213, FPR=0.0032

Class 17: TPR=0.9913, FPR=0.0397

Class 18: TPR=0.9185, FPR=0.0010

Class 19: TPR=0.9412, FPR=0.0008


In [ ]:
import random

ROUNDS = 13
CLIENTS_PER_ROUND = 9

for round_num in range(ROUNDS):

    print(f"\nFL Round {round_num}")

    global_weights = global_model.get_weights()

    client_weights = []
    client_sizes = []

    selected_clients = random.sample(
        list(client_data.keys()),
        CLIENTS_PER_ROUND
    )

    for client in selected_clients:
        weights, size = client_update(
            global_weights,
            client_data[client]
        )
        client_weights.append(weights)
        client_sizes.append(size)

    new_global_weights = aggregate_weights(client_weights, client_sizes)

    global_model.set_weights(new_global_weights)

    # Server evaluation (Approach 2)
    loss, acc = global_model.evaluate(X_server, y_server, verbose=1)
    print(f"Server Accuracy: {acc:.4f}")


FL Round 0
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 39s 8ms/step - accuracy: 0.9300 - loss: 0.0324
Server Accuracy: 0.9300

FL Round 1
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 39s 8ms/step - accuracy: 0.9259 - loss: 0.0360
Server Accuracy: 0.9259

FL Round 2
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 37s 8ms/step - accuracy: 0.9312 - loss: 0.0358
Server Accuracy: 0.9312

FL Round 3
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 37s 8ms/step - accuracy: 0.9289 - loss: 0.0347
Server Accuracy: 0.9289

FL Round 4
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 37s 8ms/step - accuracy: 0.9415 - loss: 0.0324
Server Accuracy: 0.9415

FL Round 5
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 37s 8ms/step - accuracy: 0.9339 - loss: 0.0337
Server Accuracy: 0.9339

FL Round 6
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 37s 8ms/step - accuracy: 0.9348 - loss: 0.0349
Server Accuracy: 0.9348

FL Round 7
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 37s 8ms/step - accuracy: 0.9362 - loss: 0.0339
Server Accuracy: 0.9362

FL Round 8
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 39s 8ms/step - accuracy: 0.9334 - loss: 0.033

In [ ]:
loss, acc = global_model.evaluate(X_test, y_test, verbose=1)
print("\nFinal Test Accuracy:", acc)

6080/6080 ━━━━━━━━━━━━━━━━━━━━ 47s 8ms/step - accuracy: 0.9552 - loss: 0.0298

Final Test Accuracy: 0.9551924467086792


In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)
import numpy as np

# 🔹 Get predictions
y_pred_probs = global_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# 🔹 If y_test is one-hot encoded
if len(y_test.shape) > 1:
    y_true = np.argmax(y_test, axis=1)
else:
    y_true = y_test

# 🔹 Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:\n", cm)

# 🔹 Classification Report (precision, recall, f1 per class)
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))

# 🔹 Overall Metrics
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print(f"\nPrecision: {precision:.4f}")
print(f"Recall (TPR): {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

6080/6080 ━━━━━━━━━━━━━━━━━━━━ 51s 8ms/step

Confusion Matrix:
 [[14715    43    24     6     9     1    48     0     0     0     6     7
      1     5    84    16    34     8     1     1]
 [    3 14693     5    18    11    11    31     5     1     0    53    73
      1     2     2     2    22    44     0     7]
 [    3     7  2668    54     3     0    32     4     0     0     3     7
      0     3    92     0    35     8     3     3]
 [    0    19    29  2147     8     3     0     4     0     1     1     4
      0     0     1     0    37    35     0    23]
 [    0     0    12    21   147    27     1     7     0     0     0     1
      0     2     0     0    20     1     0    18]
 [    1     2     0    34    22   137     0     1     0     0     1     0
      2     1     0     1     4     5     0     6]
 [    0     6    41     6    14     2  6922    33     0     5    25     2
      0     4    36    13    13    18     2     0]
 [    0     1     1     0     0     1    14  2195     0     0

In [ ]:
# For binary classification only
if len(np.unique(y_true)) == 2:

    tn, fp, fn, tp = cm.ravel()

    tpr = tp / (tp + fn)   # Recall / Sensitivity
    fpr = fp / (fp + tn)

    print(f"\nTPR (Recall): {tpr:.4f}")
    print(f"FPR: {fpr:.4f}")

In [ ]:
num_classes = cm.shape[0]

for i in range(num_classes):
    tp = cm[i, i]
    fn = np.sum(cm[i, :]) - tp
    fp = np.sum(cm[:, i]) - tp
    tn = np.sum(cm) - (tp + fn + fp)

    tpr = tp / (tp + fn) if (tp + fn) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0

    print(f"\nClass {i}: TPR={tpr:.4f}, FPR={fpr:.4f}")


Class 0: TPR=0.9804, FPR=0.0002

Class 1: TPR=0.9806, FPR=0.0011

Class 2: TPR=0.9121, FPR=0.0014

Class 3: TPR=0.9286, FPR=0.0016

Class 4: TPR=0.5720, FPR=0.0006

Class 5: TPR=0.6313, FPR=0.0004

Class 6: TPR=0.9692, FPR=0.0018

Class 7: TPR=0.9725, FPR=0.0004

Class 8: TPR=0.9323, FPR=0.0007

Class 9: TPR=0.8948, FPR=0.0004

Class 10: TPR=0.8944, FPR=0.0015

Class 11: TPR=0.8435, FPR=0.0032

Class 12: TPR=0.7976, FPR=0.0019

Class 13: TPR=0.7918, FPR=0.0018

Class 14: TPR=0.9559, FPR=0.0013

Class 15: TPR=0.9886, FPR=0.0004

Class 16: TPR=0.9456, FPR=0.0045

Class 17: TPR=0.9903, FPR=0.0279

Class 18: TPR=0.9191, FPR=0.0019

Class 19: TPR=0.7908, FPR=0.0007


In [ ]:
import random

ROUNDS = 13
CLIENTS_PER_ROUND = 7

for round_num in range(ROUNDS):

    print(f"\nFL Round {round_num}")

    global_weights = global_model.get_weights()

    client_weights = []
    client_sizes = []

    selected_clients = random.sample(
        list(client_data.keys()),
        CLIENTS_PER_ROUND
    )

    for client in selected_clients:
        weights, size = client_update(
            global_weights,
            client_data[client]
        )
        client_weights.append(weights)
        client_sizes.append(size)

    new_global_weights = aggregate_weights(client_weights, client_sizes)

    global_model.set_weights(new_global_weights)

    # Server evaluation (Approach 2)
    loss, acc = global_model.evaluate(X_server, y_server, verbose=1)
    print(f"Server Accuracy: {acc:.4f}")


FL Round 0
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 40s 8ms/step - accuracy: 0.9278 - loss: 0.0339
Server Accuracy: 0.9278

FL Round 1
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 41s 8ms/step - accuracy: 0.9362 - loss: 0.0340
Server Accuracy: 0.9362

FL Round 2
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 39s 8ms/step - accuracy: 0.9294 - loss: 0.0358
Server Accuracy: 0.9294

FL Round 3
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 38s 8ms/step - accuracy: 0.9348 - loss: 0.0341
Server Accuracy: 0.9348

FL Round 4
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 40s 8ms/step - accuracy: 0.9342 - loss: 0.0343
Server Accuracy: 0.9342

FL Round 5
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 39s 8ms/step - accuracy: 0.9340 - loss: 0.0342
Server Accuracy: 0.9340

FL Round 6
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 39s 8ms/step - accuracy: 0.9337 - loss: 0.0342
Server Accuracy: 0.9337

FL Round 7
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 41s 8ms/step - accuracy: 0.9417 - loss: 0.0313
Server Accuracy: 0.9417

FL Round 8
4864/4864 ━━━━━━━━━━━━━━━━━━━━ 39s 8ms/step - accuracy: 0.9483 - loss: 0.029

In [ ]:
loss, acc = global_model.evaluate(X_test, y_test, verbose=1)
print("\nFinal Test Accuracy:", acc)

6080/6080 ━━━━━━━━━━━━━━━━━━━━ 49s 8ms/step - accuracy: 0.9535 - loss: 0.0299

Final Test Accuracy: 0.9534807205200195


In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)
import numpy as np

# 🔹 Get predictions
y_pred_probs = global_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# 🔹 If y_test is one-hot encoded
if len(y_test.shape) > 1:
    y_true = np.argmax(y_test, axis=1)
else:
    y_true = y_test

# 🔹 Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:\n", cm)

# 🔹 Classification Report (precision, recall, f1 per class)
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))

# 🔹 Overall Metrics
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print(f"\nPrecision: {precision:.4f}")
print(f"Recall (TPR): {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

6080/6080 ━━━━━━━━━━━━━━━━━━━━ 58s 9ms/step

Confusion Matrix:
 [[14749    37    52     8     7     1    32     0     0     0     5    11
      1     3    66     8    20     8     1     0]
 [    2 14674     9    44    11    16    29     6     1     0    67    65
      1     2     1     3    11    38     0     4]
 [    3    14  2717    30     3     0    31     5     0     0     2     4
      0     2    84     0    14     7     4     5]
 [    0    16    50  2145    14     4     0     4     0     1     0     5
      0     3     2     2    20    26     0    20]
 [    0     0    12    25   151    28     4     7     0     0     0     1
      0     1     0     1    16     2     0     9]
 [    0     1     0    37    22   145     0     1     0     0     1     2
      0     2     0     1     1     3     0     1]
 [    0     4    42     6    13     4  6969     8     0     5    15     5
      0     3    33     5    10    17     1     2]
 [    0     2     1     0     0     1    20  2190     0     0

In [ ]:
# For binary classification only
if len(np.unique(y_true)) == 2:

    tn, fp, fn, tp = cm.ravel()

    tpr = tp / (tp + fn)   # Recall / Sensitivity
    fpr = fp / (fp + tn)

    print(f"\nTPR (Recall): {tpr:.4f}")
    print(f"FPR: {fpr:.4f}")

In [ ]:
num_classes = cm.shape[0]

for i in range(num_classes):
    tp = cm[i, i]
    fn = np.sum(cm[i, :]) - tp
    fp = np.sum(cm[:, i]) - tp
    tn = np.sum(cm) - (tp + fn + fp)

    tpr = tp / (tp + fn) if (tp + fn) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0

    print(f"\nClass {i}: TPR={tpr:.4f}, FPR={fpr:.4f}")


Class 0: TPR=0.9827, FPR=0.0003

Class 1: TPR=0.9793, FPR=0.0011

Class 2: TPR=0.9289, FPR=0.0020

Class 3: TPR=0.9278, FPR=0.0015

Class 4: TPR=0.5875, FPR=0.0007

Class 5: TPR=0.6682, FPR=0.0005

Class 6: TPR=0.9758, FPR=0.0018

Class 7: TPR=0.9703, FPR=0.0003

Class 8: TPR=0.9432, FPR=0.0010

Class 9: TPR=0.8921, FPR=0.0004

Class 10: TPR=0.8843, FPR=0.0011

Class 11: TPR=0.8746, FPR=0.0033

Class 12: TPR=0.8406, FPR=0.0032

Class 13: TPR=0.6121, FPR=0.0012

Class 14: TPR=0.9587, FPR=0.0011

Class 15: TPR=0.9826, FPR=0.0003

Class 16: TPR=0.9411, FPR=0.0035

Class 17: TPR=0.9905, FPR=0.0297

Class 18: TPR=0.9434, FPR=0.0016

Class 19: TPR=0.8411, FPR=0.0007
